### 1. Import libraries

We use NumPy, Python's `Counter`, and Hugging Face `datasets` for loading the data.

In [ ]:
import numpy as np
from collections import Counter
from datasets import load_dataset
import matplotlib.pyplot as plt

### 2. Load CoNLL-2000

The original training partition is used for training and validation; the original test partition is reserved for final evaluation.

In [ ]:
raw_dataset = load_dataset("eriktks/conll2000")

original_train_data = raw_dataset["train"]
original_test_data = raw_dataset["test"]

print("Original training sentences:", len(original_train_data))
print("Original test sentences:", len(original_test_data))
print("Available columns:", original_train_data.column_names)

### 3. Define the reproducible split

A fixed seed creates the same 90/10 sentence split each time. The returned indices let Part 1 reuse the exact split.

In [ ]:
def create_train_validation_split(training_data, validation_ratio=0.1, seed=42):
    indices = np.arange(len(training_data))

    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    validation_size = int(len(indices) * validation_ratio)
    validation_indices = indices[:validation_size].tolist()
    train_indices = indices[validation_size:].tolist()

    train_split = training_data.select(train_indices)
    validation_split = training_data.select(validation_indices)

    return train_split, validation_split, train_indices, validation_indices

### 4. Create the training and validation partitions

Use the same ratio, seed, and indices in Part 1. The test partition is not included.

In [ ]:
SEED = 42
VALIDATION_RATIO = 0.1

train_data, validation_data, train_indices, validation_indices = create_train_validation_split(
    original_train_data,
    validation_ratio=VALIDATION_RATIO,
    seed=SEED
)

test_data = original_test_data

print("Training sentences:", len(train_data))
print("Validation sentences:", len(validation_data))
print("Test sentences:", len(test_data))

# Part 2 — Hidden Markov Model and Viterbi

Implement a POS tagger using initial, transition, and emission probabilities, smoothing, an `UNK` strategy, and Viterbi decoding.

### 5. Extract tokens and POS tags

Convert POS IDs to readable tag names and prepare the original training data for final retraining. Chunk labels are not used.

In [ ]:
# Find the ClassLabel feature that contains the POS tag names.
pos_tag_feature = original_train_data.features["pos_tags"]
if hasattr(pos_tag_feature, "feature"):
    pos_tag_feature = pos_tag_feature.feature

pos_tag_names = getattr(pos_tag_feature, "names", None)


def convert_pos_tags(tag_sequence):
    if pos_tag_names is None:
        return [str(tag) for tag in tag_sequence]

    return [
        pos_tag_names[int(tag)] if isinstance(tag, (int, np.integer)) else str(tag)
        for tag in tag_sequence
    ]


def extract_sentences_and_tags(dataset_split):
    sentences = [list(tokens) for tokens in dataset_split["tokens"]]
    tag_sequences = [convert_pos_tags(tags) for tags in dataset_split["pos_tags"]]
    return sentences, tag_sequences


train_sentences, train_tags = extract_sentences_and_tags(train_data)
validation_sentences, validation_tags = extract_sentences_and_tags(validation_data)
test_sentences, test_tags = extract_sentences_and_tags(test_data)
full_train_sentences, full_train_tags = extract_sentences_and_tags(original_train_data)

### 6. Check the prepared data

Check the split sizes and inspect the extracted tokens and POS tags.

In [ ]:
print("Training sentences:", len(train_sentences))
print("Validation sentences:", len(validation_sentences))
print("Test sentences:", len(test_sentences))
print("Unique POS tags:", len({tag for sequence in train_tags for tag in sequence}))

print("\nExample sentence:", train_sentences[0])
print("Example POS tags:", train_tags[0])

assert len(train_sentences) == len(train_tags)
assert len(validation_sentences) == len(validation_tags)
assert len(test_sentences) == len(test_tags)
assert all(len(words) == len(tags) for words, tags in zip(train_sentences, train_tags))
assert all(len(words) == len(tags) for words, tags in zip(validation_sentences, validation_tags))
assert all(len(words) == len(tags) for words, tags in zip(test_sentences, test_tags))

: 

### 7. Handle unknown words

Keep capitalization and punctuation. Rare training words and unseen validation/test words are mapped to `UNK`.

In [ ]:
UNK_TOKEN = "<UNK>"

### 8. Train the HMM

Estimate initial, transition, and emission log probabilities from the training split using additive smoothing.

In [ ]:
def train_hmm(
    sentences,
    tag_sequences,
    min_word_frequency=2,
    alpha=0.1,
    beta=0.1,
    gamma=0.1
):
    # Count how often each word appears in the training sentences.
    word_frequencies = Counter(
        word for sentence in sentences for word in sentence
    )

    # Keep frequent words and reserve one vocabulary entry for unknown words.
    vocabulary = {
        word for word, count in word_frequencies.items()
        if count >= min_word_frequency
    }
    vocabulary.add(UNK_TOKEN)

    # Replace rare training words with UNK.
    processed_sentences = [
        [word if word in vocabulary else UNK_TOKEN for word in sentence]
        for sentence in sentences
    ]

    tags = sorted({tag for sequence in tag_sequences for tag in sequence})
    number_of_tags = len(tags)
    vocabulary_size = len(vocabulary)

    initial_counts = Counter()
    transition_counts = {tag: Counter() for tag in tags}
    emission_counts = {tag: Counter() for tag in tags}
    tag_counts = Counter()

    # Collect the HMM counts from the tagged training sentences.
    for words, sentence_tags in zip(processed_sentences, tag_sequences):
        if not words:
            continue

        initial_counts[sentence_tags[0]] += 1

        for word, tag in zip(words, sentence_tags):
            emission_counts[tag][word] += 1
            tag_counts[tag] += 1

        for previous_tag, current_tag in zip(sentence_tags[:-1], sentence_tags[1:]):
            transition_counts[previous_tag][current_tag] += 1

    # Initial log probabilities with additive smoothing.
    number_of_sentences = sum(initial_counts.values())
    log_initial = np.array([
        np.log(
            (initial_counts[tag] + gamma)
            / (number_of_sentences + gamma * number_of_tags)
        )
        for tag in tags
    ])

    # Transition log-probability matrix: rows = previous tags, columns = current tags.
    log_transition = np.zeros((number_of_tags, number_of_tags))

    for previous_index, previous_tag in enumerate(tags):
        outgoing_count = sum(transition_counts[previous_tag].values())
        denominator = outgoing_count + alpha * number_of_tags

        for current_index, current_tag in enumerate(tags):
            probability = (
                transition_counts[previous_tag][current_tag] + alpha
            ) / denominator
            log_transition[previous_index, current_index] = np.log(probability)

    # Store observed emission log probabilities and a fallback for zero-count words.
    log_emission = {}
    log_unk_emission = {}

    for tag in tags:
        denominator = tag_counts[tag] + beta * vocabulary_size

        log_unk_emission[tag] = np.log(beta / denominator)
        log_emission[tag] = {
            word: np.log((count + beta) / denominator)
            for word, count in emission_counts[tag].items()
        }

    return {
        "tags": tags,
        "vocabulary": vocabulary,
        "log_initial": log_initial,
        "log_transition": log_transition,
        "log_emission": log_emission,
        "log_unk_emission": log_unk_emission,
        "min_word_frequency": min_word_frequency,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma
    }

### 9. Viterbi decoding

Use dynamic programming and backpointers to find the most likely tag sequence. Log probabilities help avoid numerical underflow.

In [ ]:
def viterbi(sentence, hmm):
    if not sentence:
        return []

    tags = hmm["tags"]
    number_of_tags = len(tags)

    # Map words outside the HMM vocabulary to UNK.
    words = [
        word if word in hmm["vocabulary"] else UNK_TOKEN
        for word in sentence
    ]

    # Initialize scores for the first word.
    first_emission_scores = np.array([
        hmm["log_emission"][tag].get(
            words[0], hmm["log_unk_emission"][tag]
        )
        for tag in tags
    ])
    scores = hmm["log_initial"] + first_emission_scores

    # Store the best previous tag for each position and current tag.
    backpointers = np.zeros((len(words), number_of_tags), dtype=int)

    for position in range(1, len(words)):
        # Each row is a previous tag; each column is a current tag.
        candidate_scores = scores[:, np.newaxis] + hmm["log_transition"]
        best_previous_tags = np.argmax(candidate_scores, axis=0)

        best_previous_scores = candidate_scores[
            best_previous_tags, np.arange(number_of_tags)
        ]

        emission_scores = np.array([
            hmm["log_emission"][tag].get(
                words[position], hmm["log_unk_emission"][tag]
            )
            for tag in tags
        ])

        scores = best_previous_scores + emission_scores
        backpointers[position] = best_previous_tags

    # Choose the best tag for the final word and backtrack.
    best_last_tag = int(np.argmax(scores))
    best_path = [best_last_tag]

    for position in range(len(words) - 1, 0, -1):
        previous_tag = backpointers[position, best_path[-1]]
        best_path.append(int(previous_tag))

    best_path.reverse()
    return [tags[tag_index] for tag_index in best_path]


def predict_sentences(sentences, hmm):
    return [viterbi(sentence, hmm) for sentence in sentences]

### 10. Evaluation metrics

Report token accuracy and macro/micro precision, recall, and F1, including separate results for unknown words.

In [ ]:
def calculate_metrics(true_values, predicted_values, labels):
    if not true_values:
        return None

    correct = sum(
        true_tag == predicted_tag
        for true_tag, predicted_tag in zip(true_values, predicted_values)
    )
    accuracy = correct / len(true_values)

    per_tag = {}
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for tag in labels:
        true_positive = sum(
            true_tag == tag and predicted_tag == tag
            for true_tag, predicted_tag in zip(true_values, predicted_values)
        )
        false_positive = sum(
            true_tag != tag and predicted_tag == tag
            for true_tag, predicted_tag in zip(true_values, predicted_values)
        )
        false_negative = sum(
            true_tag == tag and predicted_tag != tag
            for true_tag, predicted_tag in zip(true_values, predicted_values)
        )

        precision = (
            true_positive / (true_positive + false_positive)
            if true_positive + false_positive > 0 else 0.0
        )
        recall = (
            true_positive / (true_positive + false_negative)
            if true_positive + false_negative > 0 else 0.0
        )
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        per_tag[tag] = {
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

        total_tp += true_positive
        total_fp += false_positive
        total_fn += false_negative

    macro_precision = np.mean([scores["precision"] for scores in per_tag.values()])
    macro_recall = np.mean([scores["recall"] for scores in per_tag.values()])
    macro_f1 = np.mean([scores["f1"] for scores in per_tag.values()])

    micro_precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp > 0 else 0.0
    )
    micro_recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn > 0 else 0.0
    )
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if micro_precision + micro_recall > 0 else 0.0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "per_tag": per_tag
    }


def evaluate_hmm(sentences, true_tag_sequences, predicted_tag_sequences, vocabulary, tags):
    all_true = []
    all_predicted = []
    unknown_true = []
    unknown_predicted = []

    for words, true_tags, predicted_tags in zip(
        sentences, true_tag_sequences, predicted_tag_sequences
    ):
        for word, true_tag, predicted_tag in zip(words, true_tags, predicted_tags):
            all_true.append(true_tag)
            all_predicted.append(predicted_tag)

            if word not in vocabulary:
                unknown_true.append(true_tag)
                unknown_predicted.append(predicted_tag)

    overall_metrics = calculate_metrics(all_true, all_predicted, tags)

    if unknown_true:
        unknown_labels = sorted(set(unknown_true) | set(unknown_predicted))
        unknown_metrics = calculate_metrics(
            unknown_true, unknown_predicted, unknown_labels
        )
    else:
        unknown_metrics = None

    return {
        "overall": overall_metrics,
        "unknown": unknown_metrics,
        "unknown_token_count": len(unknown_true)
    }


def print_metrics(results, title):
    print(title)
    print(f"Token accuracy:  {results['accuracy']:.4f}")
    print(f"Macro precision: {results['macro_precision']:.4f}")
    print(f"Macro recall:    {results['macro_recall']:.4f}")
    print(f"Macro F1:        {results['macro_f1']:.4f}")
    print(f"Micro precision: {results['micro_precision']:.4f}")
    print(f"Micro recall:    {results['micro_recall']:.4f}")
    print(f"Micro F1:        {results['micro_f1']:.4f}")

### 11. Tune on validation data

Compare vocabulary and smoothing settings using validation macro F1. The test set is not used for tuning.

In [ ]:
minimum_frequency_values = [1, 2, 3]
alpha_values = [0.1, 1.0]
beta_values = [0.1, 1.0]
gamma_values = [0.1, 1.0]

best_config = None
best_validation_f1 = -1.0
validation_results = []

for min_frequency in minimum_frequency_values:
    for alpha in alpha_values:
        for beta in beta_values:
            for gamma in gamma_values:
                model = train_hmm(
                    train_sentences,
                    train_tags,
                    min_word_frequency=min_frequency,
                    alpha=alpha,
                    beta=beta,
                    gamma=gamma
                )

                validation_predictions = predict_sentences(validation_sentences, model)
                results = evaluate_hmm(
                    validation_sentences,
                    validation_tags,
                    validation_predictions,
                    model["vocabulary"],
                    model["tags"]
                )

                validation_f1 = results["overall"]["macro_f1"]
                validation_results.append({
                    "min_word_frequency": min_frequency,
                    "alpha": alpha,
                    "beta": beta,
                    "gamma": gamma,
                    "validation_macro_f1": validation_f1
                })

                print(
                    f"min_frequency={min_frequency}, alpha={alpha}, beta={beta}, gamma={gamma}, "
                    f"validation macro F1={validation_f1:.4f}"
                )

                if validation_f1 > best_validation_f1:
                    best_validation_f1 = validation_f1
                    best_config = {
                        "min_word_frequency": min_frequency,
                        "alpha": alpha,
                        "beta": beta,
                        "gamma": gamma
                    }

print("\nBest settings:", best_config)
print(f"Best validation macro F1: {best_validation_f1:.4f}")

### 12. Validation results

Evaluate the selected configuration on the validation split, including unknown-word performance.

In [ ]:
validation_model = train_hmm(
    train_sentences,
    train_tags,
    **best_config
)

validation_predictions = predict_sentences(validation_sentences, validation_model)
validation_metrics = evaluate_hmm(
    validation_sentences,
    validation_tags,
    validation_predictions,
    validation_model["vocabulary"],
    validation_model["tags"]
)

print_metrics(validation_metrics["overall"], "Validation results — all tokens")
print("\nUnknown validation tokens:", validation_metrics["unknown_token_count"])

if validation_metrics["unknown"] is not None:
    print_metrics(validation_metrics["unknown"], "Validation results — unknown tokens")
else:
    print("No unknown validation tokens were found.")

### 13. Train the final HMM

Retrain with the selected hyperparameters using the complete original training partition.

In [ ]:
final_hmm = train_hmm(
    full_train_sentences,
    full_train_tags,
    **best_config
)

print("Number of POS tags:", len(final_hmm["tags"]))
print("HMM vocabulary size, including UNK:", len(final_hmm["vocabulary"]))
print("Selected settings:", best_config)

### 14. Final test evaluation

Evaluate once on the original test partition and report results for all tokens and unknown tokens separately.

In [ ]:
test_predictions = predict_sentences(test_sentences, final_hmm)

test_metrics = evaluate_hmm(
    test_sentences,
    test_tags,
    test_predictions,
    final_hmm["vocabulary"],
    final_hmm["tags"]
)

print_metrics(test_metrics["overall"], "Test results — all tokens")
print("\nNumber of unknown test tokens:", test_metrics["unknown_token_count"])

if test_metrics["unknown"] is not None:
    print_metrics(test_metrics["unknown"], "Test results — unknown tokens")
else:
    print("No unknown test tokens were found.")

### 15. Per-tag test results

Review precision, recall, and F1 for each POS tag.

In [ ]:
print(f"{'Tag':<10}{'Precision':>12}{'Recall':>12}{'F1':>12}")

for tag, scores in sorted(test_metrics["overall"]["per_tag"].items()):
    print(
        f"{tag:<10}"
        f"{scores['precision']:>12.4f}"
        f"{scores['recall']:>12.4f}"
        f"{scores['f1']:>12.4f}"
    )

### 16. Visualize test results

These plots summarize final test performance and highlight common tagging errors.

Compare the main evaluation metrics on the original test set.

In [ ]:
metric_names = ["Accuracy", "Macro precision", "Macro recall", "Macro F1"]
metric_values = [
    test_metrics["overall"]["accuracy"],
    test_metrics["overall"]["macro_precision"],
    test_metrics["overall"]["macro_recall"],
    test_metrics["overall"]["macro_f1"]
]

plt.figure(figsize=(8, 4))
bars = plt.bar(metric_names, metric_values)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("HMM performance on the test set")
plt.xticks(rotation=15)

for bar, value in zip(bars, metric_values):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.3f}", ha="center")

plt.tight_layout()
plt.show()

Compare accuracy on words seen in training with accuracy on unknown test words.

In [ ]:
known_correct = 0
known_total = 0
unknown_correct = 0
unknown_total = 0

for sentence, true_sequence, predicted_sequence in zip(
    test_sentences, test_tags, test_predictions
):
    for word, true_tag, predicted_tag in zip(
        sentence, true_sequence, predicted_sequence
    ):
        if word in final_hmm["vocabulary"]:
            known_total += 1
            known_correct += (true_tag == predicted_tag)
        else:
            unknown_total += 1
            unknown_correct += (true_tag == predicted_tag)

category_labels = []
category_accuracies = []
category_counts = []

if known_total:
    category_labels.append("Known words")
    category_accuracies.append(known_correct / known_total)
    category_counts.append(known_total)

if unknown_total:
    category_labels.append("Unknown words")
    category_accuracies.append(unknown_correct / unknown_total)
    category_counts.append(unknown_total)

plt.figure(figsize=(7, 4))
bars = plt.bar(category_labels, category_accuracies)
plt.ylim(0, 1)
plt.ylabel("Token accuracy")
plt.title("Test accuracy: known vs. unknown words")

for bar, accuracy, count in zip(bars, category_accuracies, category_counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        accuracy + 0.015,
        f"{accuracy:.3f} (n={count})",
        ha="center"
    )

plt.tight_layout()
plt.show()

Tags with lower F1 scores are harder for the HMM to predict.

In [ ]:
per_tag_scores = test_metrics["overall"]["per_tag"]
ordered_tags = sorted(
    per_tag_scores,
    key=lambda tag: per_tag_scores[tag]["f1"]
)
f1_values = [per_tag_scores[tag]["f1"] for tag in ordered_tags]

figure_height = max(6, len(ordered_tags) * 0.28)
plt.figure(figsize=(9, figure_height))
plt.barh(ordered_tags, f1_values)
plt.xlim(0, 1)
plt.xlabel("F1 score")
plt.ylabel("POS tag")
plt.title("Test F1 score by POS tag (lowest first)")
plt.tight_layout()
plt.show()

The confusion matrix shows which POS tags are mistaken for other tags. Rows are true tags and columns are predicted tags.

In [ ]:
all_true_tags = [tag for sequence in test_tags for tag in sequence]
all_predicted_tags = [tag for sequence in test_predictions for tag in sequence]
evaluation_tags = sorted(set(all_true_tags) | set(all_predicted_tags))
tag_to_index = {tag: index for index, tag in enumerate(evaluation_tags)}

confusion_matrix = np.zeros(
    (len(evaluation_tags), len(evaluation_tags)),
    dtype=int
)

for true_tag, predicted_tag in zip(all_true_tags, all_predicted_tags):
    confusion_matrix[tag_to_index[true_tag], tag_to_index[predicted_tag]] += 1

# Normalize each row so the chart shows proportions for each true POS tag.
row_totals = confusion_matrix.sum(axis=1, keepdims=True)
normalized_confusion_matrix = np.divide(
    confusion_matrix,
    row_totals,
    out=np.zeros_like(confusion_matrix, dtype=float),
    where=row_totals != 0
)

figure_size = max(9, len(evaluation_tags) * 0.25)
fig, ax = plt.subplots(figsize=(figure_size, figure_size * 0.85))
image = ax.imshow(normalized_confusion_matrix, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(len(evaluation_tags)))
ax.set_yticks(range(len(evaluation_tags)))
ax.set_xticklabels(evaluation_tags, rotation=90, fontsize=7)
ax.set_yticklabels(evaluation_tags, fontsize=7)
ax.set_xlabel("Predicted POS tag")
ax.set_ylabel("True POS tag")
ax.set_title("Normalized POS-tag confusion matrix")
fig.colorbar(image, ax=ax, label="Proportion")
fig.tight_layout()
plt.show()

This chart highlights the most frequent incorrect tag assignments.

In [ ]:
error_counts = Counter(
    (true_tag, predicted_tag)
    for true_tag, predicted_tag in zip(all_true_tags, all_predicted_tags)
    if true_tag != predicted_tag
)
most_common_errors = error_counts.most_common(10)

if most_common_errors:
    error_labels = [f"{true_tag} → {predicted_tag}" for (true_tag, predicted_tag), _ in most_common_errors]
    error_values = [count for _, count in most_common_errors]

    plt.figure(figsize=(8, 5))
    plt.barh(error_labels[::-1], error_values[::-1])
    plt.xlabel("Number of errors")
    plt.ylabel("True tag → predicted tag")
    plt.title("10 most common POS-tagging errors")
    plt.tight_layout()
    plt.show()
else:
    print("No tagging errors were found in the test set.")

### 17. Example tagging errors

Inspect a few incorrect predictions with their surrounding context.

In [ ]:
number_of_errors_to_show = 15
shown_errors = 0

for sentence, true_sequence, predicted_sequence in zip(
    test_sentences, test_tags, test_predictions
):
    for index, (word, true_tag, predicted_tag) in enumerate(
        zip(sentence, true_sequence, predicted_sequence)
    ):
        if true_tag != predicted_tag:
            start = max(0, index - 2)
            end = min(len(sentence), index + 3)
            context = " ".join(sentence[start:end])

            print(f"Word: {word}")
            print(f"True tag: {true_tag} | Predicted tag: {predicted_tag}")
            print(f"Context: {context}\n")

            shown_errors += 1
            if shown_errors >= number_of_errors_to_show:
                break

    if shown_errors >= number_of_errors_to_show:
        break

if shown_errors == 0:
    print("No tagging errors were found in the inspected sentences.")

## Part 2 — Summary

Record the selected hyperparameters, test metrics, unknown-word results, and notable tagging errors.